In [1]:
%%bash
pip install -q "qdrant-client[fastembed]>=1.14.2" sentence-transformers rapidfuzz pyarrow pandas tqdm

if [ ! -f /kaggle/working/qdrant_bin/qdrant ]; then
  mkdir -p /kaggle/working/qdrant_bin /kaggle/working/qdrant_storage
  wget -q https://github.com/qdrant/qdrant/releases/latest/download/qdrant-x86_64-unknown-linux-musl.tar.gz -O /tmp/qdrant.tar.gz
  tar -xzf /tmp/qdrant.tar.gz -C /kaggle/working/qdrant_bin
  chmod +x /kaggle/working/qdrant_bin/qdrant
fi

In [2]:
import os, subprocess, time, urllib.request, json

os.makedirs("/kaggle/working/qdrant_storage", exist_ok=True)
os.makedirs("/kaggle/working/artifacts", exist_ok=True)

qdrant_cfg = """
storage:
  storage_path: /kaggle/working/qdrant_storage
service:
  http_port: 6333
  grpc_port: 6334
telemetry_disabled: true
"""
with open("/kaggle/working/qdrant_config.yaml", "w") as f:
    f.write(qdrant_cfg)

qdrant_proc = subprocess.Popen(
    ["/kaggle/working/qdrant_bin/qdrant", "--config-path", "/kaggle/working/qdrant_config.yaml"],
    stdout=open("/tmp/qdrant.log", "w"),
    stderr=subprocess.STDOUT,
)

for _ in range(30):
    try:
        with urllib.request.urlopen("http://localhost:6333/healthz", timeout=1) as r:
            if r.status == 200:
                print("Qdrant server is UP on http://localhost:6333")
                break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Qdrant failed to start. Check /tmp/qdrant.log")

Qdrant server is UP on http://localhost:6333


In [3]:
import glob, re
import numpy as np
import pandas as pd

parquet_candidates = glob.glob("/kaggle/input/**/recipes.parquet", recursive=True)
assert parquet_candidates, "Could not find recipes.parquet"
RECIPES_PATH = parquet_candidates[0]
print("Loading:", RECIPES_PATH)

cols = [
    "RecipeId", "Name", "CookTime", "PrepTime", "TotalTime",
    "RecipeCategory", "Keywords", "RecipeIngredientQuantities",
    "RecipeIngredientParts", "AggregatedRating", "ReviewCount",
    "RecipeInstructions", "Images"
]
df_raw = pd.read_parquet(RECIPES_PATH, columns=cols)
print(f"Raw rows: {len(df_raw):,}")

ISO_RE = re.compile(r"^P(?:(?P<days>\d+)D)?(?:T(?:(?P<hours>\d+)H)?(?:(?P<mins>\d+)M)?(?:(?P<secs>\d+)S)?)?$")

Loading: /kaggle/input/datasets/irkaal/foodcom-recipes-and-reviews/recipes.parquet
Raw rows: 522,517


In [4]:
df_raw[0:5]

,RecipeId,Name,CookTime,PrepTime,TotalTime,RecipeCategory,Keywords,RecipeIngredientQuantities,RecipeIngredientParts,AggregatedRating,ReviewCount,RecipeInstructions,Images
0,38.0,Low-Fat Berry Blue Frozen Dessert,PT24H,PT45M,PT24H45M,Frozen Desserts,"[Dessert, Low Protein, Low Cholesterol, Health...","[4, 1⁄4, 1, 1]","[blueberries, granulated sugar, vanilla yogurt...",4.5,4.0,"[Toss 2 cups berries with sugar., Let stand fo...",[https://img.sndimg.com/food/image/upload/w_55...
1,39.0,Biryani,PT25M,PT4H,PT4H25M,Chicken Breast,"[Chicken Thigh & Leg, Chicken, Poultry, Meat, ...","[1, 4, 2, 2, 8, 1⁄4, 8, 1⁄2, 1, 1, 1⁄4, 1⁄4, 1...","[saffron, milk, hot green chili peppers, onion...",3.0,1.0,[Soak saffron in warm milk for 5 minutes and p...,[https://img.sndimg.com/food/image/upload/w_55...
2,40.0,Best Lemonade,PT5M,PT30M,PT35M,Beverages,"[Low Protein, Low Cholesterol, Healthy, Summer...","[1 1⁄2, 1, None, 1 1⁄2, None, 3⁄4]","[sugar, lemons, rind of, lemon, zest of, fresh...",4.5,10.0,"[Into a 1 quart Jar with tight fitting lid, pu...",[https://img.sndimg.com/food/image/upload/w_55...
3,41.0,Carina's Tofu-Vegetable Kebabs,PT20M,PT24H,PT24H20M,Soy/Tofu,"[Beans, Vegetable, Low Cholesterol, Weeknight,...","[12, 1, 2, 1, 10, 1, 3, 2, 2, 2, 1, 2, 1⁄2, 1⁄...","[extra firm tofu, eggplant, zucchini, mushroom...",4.5,2.0,"[Drain the tofu, carefully squeezing out exces...",[https://img.sndimg.com/food/image/upload/w_55...
4,42.0,Cabbage Soup,PT30M,PT20M,PT50M,Vegetable,"[Low Protein, Vegan, Low Cholesterol, Healthy,...","[46, 4, 1, 2, 1]","[plain tomato juice, cabbage, onion, carrots, ...",4.5,11.0,"[Mix everything together and bring to a boil.,...",[https://img.sndimg.com/food/image/upload/w_55...


In [5]:
def parse_iso_minutes(val) -> int | None:
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return None
    s = str(val).strip()
    m = ISO_RE.match(s)
    if not m:
        return None
    days = int(m.group("days") or 0)
    hours = int(m.group("hours") or 0)
    mins = int(m.group("mins") or 0)
    secs = int(m.group("secs") or 0)
    total = days * 1440 + hours * 60 + mins + (1 if secs >= 30 else 0)
    return total if total > 0 else None

def to_py_list(val) -> list[str]:
    if val is None:
        return []
    if isinstance(val, (list, tuple, np.ndarray)):
        return [str(x).strip() for x in val if x is not None and str(x).strip()]
    return []

In [6]:
df = df_raw.copy()
df["total_minutes"] = df["TotalTime"].apply(parse_iso_minutes)
df["ingredients_raw_parts"] = df["RecipeIngredientParts"].apply(to_py_list)
df["quantities_raw"] = df["RecipeIngredientQuantities"].apply(to_py_list)
df["keywords_list"] = df["Keywords"].apply(to_py_list)
df["steps_list"] = df["RecipeInstructions"].apply(to_py_list)
df["images_list"] = df["Images"].apply(to_py_list)